# 05 - Evaluation for NLP: Accuracy, F1, BLEU, ROUGE & Perplexity

---

## Learning Objectives

By the end of this notebook you will be able to:

- Compute **classification metrics**: accuracy, precision, recall, F1 (micro/macro/weighted).
- Implement and interpret **BLEU** for machine translation evaluation.
- Implement and interpret **ROUGE** (ROUGE-1, ROUGE-2, ROUGE-L) for summarization.
- Calculate **perplexity** and understand its role in evaluating language models.
- Choose the right metric for your NLP task.

## Prerequisites

- Completed Notebooks 01-04 (text preprocessing, BoW/TF-IDF, tokenization).
- Basic probability and logarithms.
- Familiarity with sklearn's classification API.

## Table of Contents

1. [Classification Metrics: Accuracy, Precision, Recall, F1](#1-classification-metrics-accuracy-precision-recall-f1)
2. [BLEU: Bilingual Evaluation Understudy](#2-bleu-bilingual-evaluation-understudy)
3. [ROUGE: Recall-Oriented Understudy for Gisting Evaluation](#3-rouge-recall-oriented-understudy-for-gisting-evaluation)
4. [Perplexity: Language Model Evaluation](#4-perplexity-language-model-evaluation)
5. [Code: Classification Metrics with sklearn](#5-code-classification-metrics-with-sklearn)
6. [Code: Implement Simple BLEU](#6-code-implement-simple-bleu)
7. [Code: Implement Simple ROUGE](#7-code-implement-simple-rouge)
8. [Code: Perplexity Calculation](#8-code-perplexity-calculation)
9. [Choosing the Right Metric](#9-choosing-the-right-metric)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)
11. [Exercises](#11-exercises)
12. [Exercise Solutions](#12-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import numpy as np
from collections import Counter
from typing import List, Dict, Tuple
import math

print("Setup complete.")

## 1. Classification Metrics: Accuracy, Precision, Recall, F1

### The Confusion Matrix

For binary classification:

|  | Predicted Positive | Predicted Negative |
|--|:--:|:--:|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### Metrics

- **Accuracy**: $\frac{TP + TN}{TP + TN + FP + FN}$ -- overall correctness
- **Precision**: $\frac{TP}{TP + FP}$ -- of predicted positives, how many are correct?
- **Recall** (Sensitivity): $\frac{TP}{TP + FN}$ -- of actual positives, how many did we find?
- **F1 Score**: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ -- harmonic mean of precision and recall

### Multi-class Averaging

| Averaging | Formula | When to Use |
|-----------|---------|------------|
| **Micro** | Global TP, FP, FN | All classes equally important |
| **Macro** | Average per-class metrics | Each class equally important (regardless of size) |
| **Weighted** | Size-weighted average | Account for class imbalance |

### When Accuracy Fails

With 95% negative, 5% positive: predicting "all negative" gives 95% accuracy but 0% recall on positives. Use F1 for imbalanced datasets.

In [ ]:
# ---- Classification metrics from scratch ----

def compute_classification_metrics(y_true: np.ndarray, y_pred: np.ndarray
                                   ) -> Dict[str, float]:
    """Compute accuracy, precision, recall, F1 for binary classification."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }

# Example: Sentiment analysis predictions
np.random.seed(42)
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0])
y_pred = np.array([1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0])

metrics = compute_classification_metrics(y_true, y_pred)
print("Binary Classification Metrics (from scratch):")
print(f"  Confusion: TP={metrics['tp']}, TN={metrics['tn']}, "
      f"FP={metrics['fp']}, FN={metrics['fn']}")
print(f"  Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1 Score:  {metrics['f1']:.4f}")

## 2. BLEU: Bilingual Evaluation Understudy

BLEU measures how similar a **candidate** (machine-generated) translation is to one or more **reference** translations.

### Formula

$$\text{BLEU} = \text{BP} \times \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where:

- $p_n$ = **modified n-gram precision**: fraction of candidate n-grams found in the reference (clipped to reference counts)
- $w_n = \frac{1}{N}$ typically (uniform weights, $N=4$)
- $\text{BP}$ = **brevity penalty**: penalizes candidates shorter than the reference

$$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

where $c$ = candidate length, $r$ = reference length.

### Key Properties

- **Precision-oriented**: Measures how much of the candidate appears in the reference.
- **Brevity penalty**: Prevents gaming by generating very short outputs.
- **Range**: 0 to 1 (often reported as 0-100).
- **Higher is better**.
- Standard for **machine translation** evaluation.

## 3. ROUGE: Recall-Oriented Understudy for Gisting Evaluation

ROUGE measures overlap between a **candidate** summary and a **reference** summary.

### Variants

| Variant | What it measures |
|---------|------------------|
| **ROUGE-1** | Unigram overlap |
| **ROUGE-2** | Bigram overlap |
| **ROUGE-L** | Longest Common Subsequence (LCS) |

### ROUGE-N Formula

$$\text{ROUGE-N}_{\text{recall}} = \frac{\text{count of matching n-grams}}{\text{count of n-grams in reference}}$$

$$\text{ROUGE-N}_{\text{precision}} = \frac{\text{count of matching n-grams}}{\text{count of n-grams in candidate}}$$

$$\text{ROUGE-N}_{\text{F1}} = 2 \times \frac{P \times R}{P + R}$$

### ROUGE-L

Uses Longest Common Subsequence (LCS). Does not require consecutive matches:

$$R_{\text{LCS}} = \frac{\text{LCS}(\text{ref}, \text{cand})}{|\text{ref}|}$$

$$P_{\text{LCS}} = \frac{\text{LCS}(\text{ref}, \text{cand})}{|\text{cand}|}$$

### Key Properties

- **Recall-oriented**: How much of the reference is captured?
- **Higher is better**.
- Standard for **text summarization** evaluation.

## 4. Perplexity: Language Model Evaluation

Perplexity measures how well a language model predicts a test corpus.

### Formula

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i | w_{<i})\right)$$

where:
- $N$ = number of tokens
- $P(w_i | w_{<i})$ = model's predicted probability for token $w_i$ given preceding context

### Intuition

- Perplexity is the **exponential of the average negative log-likelihood**.
- It represents the "effective number of equally likely choices" the model faces at each step.
- **Lower is better**: PPL=1 means the model is perfectly certain; PPL=1000 means high uncertainty.

### Typical Values

| Model | Perplexity (approximate) |
|-------|:------------------------:|
| Random (50K vocab) | ~50,000 |
| Basic n-gram | ~200-400 |
| LSTM language model | ~50-100 |
| GPT-2 (small) | ~30-40 |
| GPT-3 | ~15-25 |

### Important Caveat

Perplexity can only be compared across models that use the **same vocabulary/tokenization**. A model with 30K vocab and one with 50K vocab have different perplexity baselines.

## 5. Code: Classification Metrics with sklearn

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ---- Multi-class example: Topic classification ----
# 0=sports, 1=politics, 2=technology
y_true_mc = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2])
y_pred_mc = np.array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 2, 2, 2, 2, 1, 2, 2, 2, 0])

class_names = ["sports", "politics", "technology"]

# Confusion matrix
cm = confusion_matrix(y_true_mc, y_pred_mc)
print("Confusion Matrix:")
print(f"{'':>12}", end="")
for name in class_names:
    print(f"{name:>12}", end="")
print()
for i, name in enumerate(class_names):
    print(f"{name:>12}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i][j]:>12}", end="")
    print()
print()

# Classification report
print("Classification Report:")
print(classification_report(y_true_mc, y_pred_mc, target_names=class_names))

In [ ]:
# ---- Compare micro, macro, weighted F1 ----

for avg in ["micro", "macro", "weighted"]:
    f1 = f1_score(y_true_mc, y_pred_mc, average=avg)
    print(f"  F1 ({avg:8s}): {f1:.4f}")

print()
print("Interpretation:")
print("  - Micro: treats each sample equally (= accuracy for multi-class).")
print("  - Macro: treats each CLASS equally (sensitive to minority classes).")
print("  - Weighted: accounts for class sizes.")

In [ ]:
# ---- Demonstrate accuracy failure on imbalanced data ----

# 95% negative, 5% positive
np.random.seed(42)
n = 200
y_true_imb = np.array([1]*10 + [0]*190)
y_pred_all_neg = np.zeros(n, dtype=int)  # predict everything as negative

print("Imbalanced Dataset (5% positive, 95% negative):")
print(f"  Always-negative classifier:")
print(f"    Accuracy:  {accuracy_score(y_true_imb, y_pred_all_neg):.4f}  (misleadingly high!)")
print(f"    Precision: {precision_score(y_true_imb, y_pred_all_neg, zero_division=0):.4f}")
print(f"    Recall:    {recall_score(y_true_imb, y_pred_all_neg):.4f}  (zero -- misses ALL positives!)")
print(f"    F1:        {f1_score(y_true_imb, y_pred_all_neg):.4f}  (correctly shows poor performance)")
print()
print("Lesson: For imbalanced data, F1 is much more informative than accuracy.")

## 6. Code: Implement Simple BLEU

In [ ]:
def get_ngrams(tokens: List[str], n: int) -> Counter:
    """Extract n-grams from a token list and count them."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def modified_precision(candidate: List[str], reference: List[str],
                       n: int) -> float:
    """Compute modified n-gram precision (clipped counts)."""
    cand_ngrams = get_ngrams(candidate, n)
    ref_ngrams = get_ngrams(reference, n)
    
    if not cand_ngrams:
        return 0.0
    
    # Clip candidate counts to reference counts
    clipped_count = 0
    for ngram, count in cand_ngrams.items():
        clipped_count += min(count, ref_ngrams.get(ngram, 0))
    
    total_count = sum(cand_ngrams.values())
    return clipped_count / total_count if total_count > 0 else 0.0


def compute_bleu(candidate: List[str], reference: List[str],
                 max_n: int = 4) -> Dict[str, float]:
    """Compute BLEU score.
    
    Args:
        candidate: Tokenized candidate translation.
        reference: Tokenized reference translation.
        max_n: Maximum n-gram order (default 4 for BLEU-4).
    
    Returns:
        Dictionary with BLEU score and component details.
    """
    # Compute modified precision for each n
    precisions = []
    for n in range(1, max_n + 1):
        p = modified_precision(candidate, reference, n)
        precisions.append(p)
    
    # Brevity penalty
    c = len(candidate)
    r = len(reference)
    if c > r:
        bp = 1.0
    elif c == 0:
        bp = 0.0
    else:
        bp = math.exp(1 - r / c)
    
    # Geometric mean of precisions (using log to avoid underflow)
    # Only include n-grams where precision > 0
    log_precisions = []
    for p in precisions:
        if p > 0:
            log_precisions.append(math.log(p))
        else:
            log_precisions.append(float('-inf'))
    
    if any(lp == float('-inf') for lp in log_precisions):
        bleu = 0.0
    else:
        w = 1.0 / max_n  # uniform weights
        bleu = bp * math.exp(sum(w * lp for lp in log_precisions))
    
    return {
        "bleu": bleu,
        "brevity_penalty": bp,
        "precisions": precisions,
        "candidate_len": c,
        "reference_len": r,
    }


# ---- Example: Machine translation evaluation ----

reference = "the cat is on the mat".split()
candidate1 = "the cat is on the mat".split()     # perfect match
candidate2 = "the the the the the the".split()   # gaming with repeated words
candidate3 = "a cat sits on a mat".split()        # paraphrase
candidate4 = "cat mat".split()                     # too short

print("BLEU Score Examples:")
print(f"  Reference: {' '.join(reference)}")
print("=" * 65)

for name, cand in [("Perfect match", candidate1),
                    ("Repeated words", candidate2),
                    ("Paraphrase", candidate3),
                    ("Too short", candidate4)]:
    result = compute_bleu(cand, reference, max_n=4)
    print(f"\n  Candidate ({name}): {' '.join(cand)}")
    print(f"    BLEU:     {result['bleu']:.4f}")
    print(f"    BP:       {result['brevity_penalty']:.4f}")
    print(f"    Prec 1-4: {[f'{p:.3f}' for p in result['precisions']]}")

## 7. Code: Implement Simple ROUGE

In [ ]:
def compute_rouge_n(candidate: List[str], reference: List[str],
                    n: int = 1) -> Dict[str, float]:
    """Compute ROUGE-N (precision, recall, F1).
    
    Args:
        candidate: Tokenized candidate summary.
        reference: Tokenized reference summary.
        n: N-gram order (1 for ROUGE-1, 2 for ROUGE-2).
    """
    cand_ngrams = get_ngrams(candidate, n)
    ref_ngrams = get_ngrams(reference, n)
    
    # Count matching n-grams (clipped)
    matches = 0
    for ngram, count in ref_ngrams.items():
        matches += min(count, cand_ngrams.get(ngram, 0))
    
    ref_total = sum(ref_ngrams.values())
    cand_total = sum(cand_ngrams.values())
    
    recall = matches / ref_total if ref_total > 0 else 0.0
    precision = matches / cand_total if cand_total > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    
    return {"precision": precision, "recall": recall, "f1": f1}


def lcs_length(a: List[str], b: List[str]) -> int:
    """Compute length of Longest Common Subsequence."""
    m, n = len(a), len(b)
    # DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


def compute_rouge_l(candidate: List[str], reference: List[str]
                    ) -> Dict[str, float]:
    """Compute ROUGE-L using Longest Common Subsequence."""
    lcs_len = lcs_length(reference, candidate)
    
    recall = lcs_len / len(reference) if len(reference) > 0 else 0.0
    precision = lcs_len / len(candidate) if len(candidate) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    
    return {"precision": precision, "recall": recall, "f1": f1, "lcs": lcs_len}


# ---- Example: Summarization evaluation ----

reference = "the cat sat on the mat near the window".split()
candidate1 = "the cat sat on the mat near the window".split()  # perfect
candidate2 = "the cat was on the mat".split()                   # partial match
candidate3 = "a dog stood on a rug by the door".split()         # poor match

print("ROUGE Score Examples:")
print(f"  Reference: {' '.join(reference)}")
print("=" * 70)

for name, cand in [("Perfect match", candidate1),
                    ("Partial match", candidate2),
                    ("Poor match", candidate3)]:
    r1 = compute_rouge_n(cand, reference, n=1)
    r2 = compute_rouge_n(cand, reference, n=2)
    rl = compute_rouge_l(cand, reference)
    
    print(f"\n  Candidate ({name}): {' '.join(cand)}")
    print(f"    ROUGE-1 F1: {r1['f1']:.4f}  (P={r1['precision']:.3f}, R={r1['recall']:.3f})")
    print(f"    ROUGE-2 F1: {r2['f1']:.4f}  (P={r2['precision']:.3f}, R={r2['recall']:.3f})")
    print(f"    ROUGE-L F1: {rl['f1']:.4f}  (LCS={rl['lcs']})")

## 8. Code: Perplexity Calculation

In [ ]:
def compute_perplexity(log_probs: np.ndarray) -> float:
    """Compute perplexity from log probabilities.
    
    PPL = exp(-1/N * sum(log P(w_i)))
    
    Args:
        log_probs: Array of log probabilities (natural log) for each token.
    
    Returns:
        Perplexity value.
    """
    N = len(log_probs)
    avg_neg_log_prob = -np.mean(log_probs)
    return float(np.exp(avg_neg_log_prob))


# ---- Example: Comparing language models ----

# Simulate model predictions on a sequence of 10 tokens
# Each value is log P(w_i | context)

# Good model: assigns high probabilities (log probs close to 0)
np.random.seed(42)
good_model_log_probs = np.log(np.random.uniform(0.3, 0.9, size=10))

# Mediocre model: moderate probabilities
mediocre_model_log_probs = np.log(np.random.uniform(0.05, 0.3, size=10))

# Bad model: low probabilities
bad_model_log_probs = np.log(np.random.uniform(0.001, 0.05, size=10))

# Uniform random baseline with vocab size 10000
uniform_log_probs = np.full(10, np.log(1.0 / 10000))

print("Perplexity Comparison:")
print("=" * 50)
for name, log_probs in [
    ("Good model", good_model_log_probs),
    ("Mediocre model", mediocre_model_log_probs),
    ("Bad model", bad_model_log_probs),
    ("Uniform random (V=10K)", uniform_log_probs),
]:
    ppl = compute_perplexity(log_probs)
    avg_prob = np.exp(np.mean(log_probs))
    print(f"  {name:<25s}  PPL = {ppl:>10.2f}  "
          f"(avg prob = {avg_prob:.4f})")

print()
print("Lower perplexity = better model.")
print("PPL ~ V (vocab size) for uniform random = worst case.")

In [ ]:
# ---- Perplexity from a simple bigram language model ----

def build_bigram_model(corpus: List[str]) -> Dict[str, Dict[str, float]]:
    """Build a bigram language model with add-1 (Laplace) smoothing."""
    # Count bigrams and unigrams
    bigram_counts = Counter()
    unigram_counts = Counter()
    vocab = set()
    
    for text in corpus:
        tokens = ["<s>"] + text.lower().split() + ["</s>"]
        vocab.update(tokens)
        for i in range(len(tokens) - 1):
            bigram_counts[(tokens[i], tokens[i+1])] += 1
            unigram_counts[tokens[i]] += 1
    
    V = len(vocab)
    
    # Compute probabilities with Laplace smoothing
    model = {}
    for (w1, w2), count in bigram_counts.items():
        if w1 not in model:
            model[w1] = {}
        model[w1][w2] = (count + 1) / (unigram_counts[w1] + V)
    
    return model, V, unigram_counts


def bigram_perplexity(model, V, unigram_counts, text: str) -> float:
    """Compute perplexity of a text under the bigram model."""
    tokens = ["<s>"] + text.lower().split() + ["</s>"]
    log_probs = []
    
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]
        if w1 in model and w2 in model[w1]:
            prob = model[w1][w2]
        else:
            # Laplace smoothing fallback
            prob = 1 / (unigram_counts.get(w1, 0) + V)
        log_probs.append(np.log(prob))
    
    return compute_perplexity(np.array(log_probs))


# Train on a small corpus
train_corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the dog",
    "the dog chased the cat",
    "a cat sat on a mat",
]

bigram_model, V, uni_counts = build_bigram_model(train_corpus)

# Test sentences
test_sentences = [
    "the cat sat on the mat",     # seen in training
    "the dog sat on the mat",     # plausible
    "a cat chased the dog",       # plausible
    "banana quantum helicopter",  # nonsensical
]

print("Bigram Language Model Perplexity:")
print("=" * 60)
for sent in test_sentences:
    ppl = bigram_perplexity(bigram_model, V, uni_counts, sent)
    print(f"  PPL = {ppl:>8.2f}  | {sent}")

print()
print("In-domain sentences have lower perplexity (more predictable).")
print("Out-of-domain/nonsensical text has much higher perplexity.")

## 9. Choosing the Right Metric

| NLP Task | Primary Metrics | Why |
|----------|----------------|-----|
| **Text classification** | F1, Accuracy | Direct measure of correct labels |
| **Named Entity Recognition** | Entity-level F1 | Partial matches need special handling |
| **Machine translation** | BLEU, METEOR, BERTScore | Measure translation quality |
| **Summarization** | ROUGE-1, ROUGE-2, ROUGE-L | Measure content coverage |
| **Language modeling** | Perplexity | Measure prediction quality |
| **Question answering** | Exact Match (EM), F1 | Measure answer correctness |
| **Text generation** | Human eval, Perplexity, BERTScore | No single automatic metric is sufficient |

## 10. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using accuracy on imbalanced data | Misleadingly high score | Use F1 (macro or weighted) |
| Computing BLEU on untokenized text | Inflated/deflated scores | Tokenize consistently before computing |
| Comparing perplexity across different vocabularies | Unfair comparison (larger vocab = higher PPL baseline) | Only compare same-tokenizer models |
| Using BLEU for summarization | BLEU is precision-oriented; summaries need recall | Use ROUGE for summarization |
| Using ROUGE for translation | ROUGE ignores word order more than BLEU | Use BLEU for translation |
| Reporting BLEU without brevity penalty | Short outputs get unfairly high scores | Always include BP in BLEU calculation |
| Mixing up log bases | Factor-of-2 errors in perplexity | Use natural log consistently; $\text{PPL} = 2^{H}$ if using $\log_2$ |
| Not specifying F1 averaging method | Ambiguous results | Always state micro/macro/weighted |

**Debugging tip:** Always compute multiple metrics. If BLEU is high but human evaluation is poor, your metric may not capture the actual quality dimension you care about.

## 11. Exercises

### Exercise 1: Evaluate a Model with Multiple Metrics

A text classification model has produced the predictions below. Compute all relevant classification metrics and interpret the results.

In [ ]:
# Multi-class: 0=negative, 1=neutral, 2=positive
# Notice the class imbalance: many neutral, few negative
ex_y_true = np.array([2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 2, 2,
                      1, 1, 1, 2, 2, 0, 0, 1, 1, 1])
ex_y_pred = np.array([2, 2, 1, 2, 2, 1, 1, 1, 0, 1, 1, 1, 1, 2, 1, 0, 1, 0, 2, 1,
                      1, 1, 0, 2, 2, 0, 0, 1, 2, 1])

# TODO:
# 1. Print the confusion matrix
# 2. Print the full classification report
# 3. Compute micro, macro, and weighted F1
# 4. Which class has the worst performance? Why?
# Your code here

### Exercise 2: BLEU and ROUGE Comparison

Compute BLEU and ROUGE for the candidates below and explain why the two metrics may disagree.

In [ ]:
reference = "the quick brown fox jumps over the lazy dog".split()

candidates = {
    "A (verbatim)": "the quick brown fox jumps over the lazy dog".split(),
    "B (reordered)": "the lazy dog the quick brown fox jumps over".split(),
    "C (shorter)":   "the quick brown fox jumps".split(),
    "D (longer)":    "the quick brown fox jumps over the lazy dog today in the park".split(),
}

# TODO:
# 1. Compute BLEU (max_n=4) and ROUGE-1/2/L for each candidate
# 2. Which metric favors which candidate?
# 3. Why might BLEU and ROUGE disagree?
# Your code here

## 12. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----

print("Exercise 1: Multi-class Classification Evaluation")
print("=" * 60)

class_names = ["negative", "neutral", "positive"]

# Confusion matrix
cm = confusion_matrix(ex_y_true, ex_y_pred)
print("\nConfusion Matrix:")
print(f"{'':>12}", end="")
for name in class_names:
    print(f"{name:>12}", end="")
print()
for i, name in enumerate(class_names):
    print(f"{name:>12}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# Classification report
print("\nClassification Report:")
print(classification_report(ex_y_true, ex_y_pred, target_names=class_names))

# F1 variants
print("F1 Score Variants:")
for avg in ["micro", "macro", "weighted"]:
    f1 = f1_score(ex_y_true, ex_y_pred, average=avg)
    print(f"  F1 ({avg:8s}): {f1:.4f}")

print()
print("Analysis:")
print("  - Neutral (majority class) has best F1 due to more training signal.")
print("  - Negative (minority) may have lower recall since few samples exist.")
print("  - Macro F1 < Weighted F1 when minority class performs worse.")

In [ ]:
# ---- Exercise 2 Solution ----

print("Exercise 2: BLEU vs ROUGE Comparison")
print("=" * 70)
print(f"Reference: {' '.join(reference)}")
print()

for name, cand in candidates.items():
    bleu_result = compute_bleu(cand, reference, max_n=4)
    r1 = compute_rouge_n(cand, reference, n=1)
    r2 = compute_rouge_n(cand, reference, n=2)
    rl = compute_rouge_l(cand, reference)
    
    print(f"Candidate {name}: {' '.join(cand)}")
    print(f"  BLEU:      {bleu_result['bleu']:.4f}  (BP={bleu_result['brevity_penalty']:.3f})")
    print(f"  ROUGE-1:   {r1['f1']:.4f}  (P={r1['precision']:.3f}, R={r1['recall']:.3f})")
    print(f"  ROUGE-2:   {r2['f1']:.4f}  (P={r2['precision']:.3f}, R={r2['recall']:.3f})")
    print(f"  ROUGE-L:   {rl['f1']:.4f}")
    print()

print("Key differences:")
print("  - BLEU is precision-focused: penalizes adding extra words less.")
print("  - ROUGE is recall-focused: penalizes missing reference content.")
print("  - Candidate C (shorter): high ROUGE recall may drop, but BLEU BP also penalizes.")
print("  - Candidate D (longer): ROUGE recall stays high, BLEU precision may drop.")
print("  - Candidate B (reordered): ROUGE-1 unaffected (no order), BLEU drops (n-gram order matters).")